In [ ]:
using FFTW
using DifferentialEquations
using Plots

In [ ]:

# KdV: u_t + u u_x + u_xxx = 0 on a periodic domain x in [0, L)
# These defaults are chosen to run quickly in a notebook; increase N/tspan for higher fidelity.
N = 1024
L = 4π
x = L .* (0:N-1) ./ N

# Fourier wave numbers
k = (2π / L) .* vcat(0:N÷2, -N÷2+1:-1)
ik = 1im .* k
# ik3 = ik .^ 3
ik3 = 1im .* k.^3

function kdv_if_pseudospectral!(dvhat, vhat, p, t)
    ik, ik3 = p

    Eminus = @. exp(-ik3 * t);
    Eplus = @. exp(ik3 * t);
    uhat = @. Eplus * vhat;

    u = real(ifft(uhat));

    nlhat = fft(u.^2);
    # nlhat[dealias] .= 0
    @. dvhat = -Eminus * ik * 0.5 * nlhat;
    dvhat
end


In [ ]:

# Periodic initial condition
# u0 = @. exp(-4 * (x - L/2)^2)
A = 25; B = 16;
u0 = @. 3 * A^2 * sech(.5 *(A* (x - L/2+2)))^2 +  3 * B^2 * sech(.5 *(B *(x - L/2+1)))^2 
vhat0 = fft(u0)

tspan = (0.0, 0.01)
# p = (ik, ik3, dealias)
p = (ik, ik3)

prob = ODEProblem(kdv_if_pseudospectral!, vhat0, tspan, p)

# DifferentialEquations.jl is used for time stepping.
sol = solve(prob; saveat=0.0001)

In [ ]:

# Reconstruct u(x,t) from integrating-factor variable v̂
Ucols = [real.(ifft(exp.(ik3 .* t) .* vhat)) for (t, vhat) in zip(sol.t, sol.u)]
U = reduce(hcat, Ucols)  # size (N, nt)

contourf(
    x, sol.t, U',
    xlabel="x", ylabel="t",
    title="KdV solution via pseudospectral method",
    c=:viridis, colorbar=true
    )

In [ ]:
anim = @animate for i in 1:length(sol.t)
    plot(x, U[:, i], ylim=(-250, 2500), 
    title="t = $(round(sol.t[i], digits=6))", label="")
    xlabel!("x")
    ylabel!("u")
end
gif(anim, fps=6)